# DESC ELAsTiCC2 — Demo 1 : Light-curve fitting with Bazin curve with function implemented in the notebook

- author : Sylvie Dagoret-Campagne
- creation date : 2026-05-09

## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.special import gamma as gamma_func


# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")



- article : **Photometric classi ation and redshift estimation of LSST Supernovae**
- authors : Mi Dai, Steve Kuhlmann,Yun Wang, and Eve Kovacs
- link : https://arxiv.org/pdf/1701.05689

Paramètres configurables (cellule Parameters) :

    - N_CURVES : nombre de light curves à afficher
    - Z_MIN, Z_MAX : intervalle de redshift pour la sélection
    - OBJ_CLASS : classe d'objet SNANA (ex. 'SNIa-SALT3')
    - FILE_NUM : numéro du fichier PHOT à charger (1–40, None = tous)
    - MIN_DETECTIONS : nombre minimum de détections exigé par objet
    - DETECTED_ONLY : si True, n'affiche que les points détectés (PHOTFLAG & photflag_detect != 0)
    - NCOLS : nombre de colonnes dans la grille de subplots
    - RANDOM_SEED : graine pour la reproductibilité (None = aléatoire)


In [ ]:
# ── Paramètres principaux ────────────────────────────────────────────────────

N_CURVES        = 50          # Nombre de light curves à afficher
OBJ_CLASS       = 'SNIa-SALT3'  # Classe SNANA à utiliser
Z_MIN           = 0.1         # Borne inférieure du redshift
Z_MAX           = 1.5         # Borne supérieure du redshift
FILE_NUM        = 1           # Fichier PHOT à charger (1–40 ; None = tous)
MIN_DETECTIONS  = 5           # Nombre minimum de détections par objet
DETECTED_ONLY   = True        # True : points détectés uniquement
NCOLS           = 3           # Nombre de colonnes dans la grille
RANDOM_SEED     = 42          # None pour aléatoire

# Chemin vers les données ELAsTiCC2
DATA_DIR    = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX  = "ELASTICC2_TRAIN_02_"

# ────────────────────────────────────────────────────────────────────────────
NROWS = math.ceil(N_CURVES / NCOLS)
rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"Grille : {NROWS} lignes × {NCOLS} colonnes pour {N_CURVES} light curves")


In [ ]:
MIN_BANDS = 3       # Nombre minimal de bandes avec assez de points
MIN_POINTS = 3      # Nombre minimal de points par bande
MIN_TOTAL_POINTS = MIN_DETECTIONS   # Nombre minimal total de points (toutes bandes confondues)

## 1. Ajustement par bande : Principe général


### Objectif :

La fonction de Bazin est utilisée pour modéliser la forme de la courbe de lumière de manière indépendante du modèle physique (contrairement à SALT2, qui est spécifique aux SNe Ia).

Cela permet d’extraire des paramètres génériques (t_rise, t_fall, A, B, t₀) pour chaque bande, qui serviront ensuite de features pour la classification (via Random Forest) ou pour calculer les couleurs au pic (ex : c_ug = m_p,u - m_p,g).



$$
f(t)=A_\star \exp[−(t−t_0)/t_{fall}]/[1+exp[−(t−t_0)/t_{rise]}]+B
$$

- $t_{rise}$ : Temps caractéristique de la montée en luminosité.
- $t_{fall}$ : Temps caractéristique de la décroissance.
- $A$ : Amplitude (liée à la luminosité au pic).
- $B$ : Offset (niveau de fond, ex : flux résiduel).
- $t_0$ : Temps de référence (proche du pic, mais pas exactement t_max).

## 2. Méthodologie d'ajustement par bande

### Cas 1 : Bande avec SNR > 5


#### Ajustement complet :

La fonction de Bazin (Eq. 2) est ajustée directement sur les données de la bande (flux en fonction du temps).

Outils : Utilisation de scipy.optimize.curve_fit (Python) en 2 étapes :

Premier fit : Avec des valeurs initiales larges (ex : t_fall=15, t_rise=5, A=flux au pic, B=0).
Deuxième fit : Avec des valeurs initiales basées sur la médiane des paramètres du premier fit (pour les bandes réussies).

Critère de succès : t_fall > t_rise > 1 (pour éviter des fits non physiques).


#### Calcul de la magnitude au pic (m_p) :

Une fois les paramètres ajustés, 
- on calcule le temps du pic (Eq. 3):
$$
t_{max} = t_0 + t_{rise} \times  \ln(t_{fall} / t_{rise} - 1)
$$

- le Flux au pic (Eq. 4):

$$
f_{max} = A * x^x * (1 - x)^(1 - x) + B, \text {où }x = t_{rise} / t_{fall}  
$$

- la Magnitude au pic (Eq. 5):
$$
m_p = -2.5 \times \log_{10}(f_{max}) + zero\_point  
$$ 

### Cas 2 : Bande avec SNR ≤ 5

#### Ajustement constant :

Si le SNR est trop faible (ex : bande u à haut redshift ou bande Y bruyante), la courbe est ajustée par une constante :
f(t)=Bf(t) = B
f(t)=B

Dans ce cas, t_rise, t_fall, et m_p sont mis à 0 (ou ignorés).
Les couleurs (c_ij) sont calculées en utilisant les erreurs pondérées (voir Section 3.1 de l’article).


## 3. Pourquoi un ajustement par bande ?


### Indépendance des bandes :

Les courbes de lumière dans différentes bandes (u, g, r, etc.) ont des formes distinctes (ex : le pic en bande u est plus étroit qu’en bande r).

Un ajustement global (multi-bandes) avec un modèle comme SALT2 impose des corrélations entre les bandes (ex : relation couleur-stretch), ce qui peut biaiser les features pour les SNe non-Ia (CC).


### Avantages pour la classification :

Non-biaisé pour les SNe CC : La fonction de Bazin ne suppose pas de relation entre les paramètres des différentes bandes (contrairement à SALT2).
Features robustes : Les paramètres (t_rise, t_fall) et les couleurs (c_ug, c_gr, etc.) sont calculés sans hypothèse sur le type de SN, ce qui est crucial pour un classifieur comme Random Forest.


### Inconvénients :

Perte de cohérence physique : Les paramètres ajustés indépendamment peuvent ne pas refléter une réalité physique (ex : t_max peut varier légèrement entre les bandes).
Sensibilité au bruit : Les bandes avec un faible SNR (ex : u à z > 0.6) sont mal ajustées → nécessité de coupures qualité.



## 4. Exemple concret (Tableau 1 de l'article)
L’article détaille les valeurs initiales et limites pour les 2 étapes d’ajustement :


| Paramètre | 1ère étape (initial) | 1ère étape (limites) | 2ème étape (initial) | 2ème étape (limites) |
| --- | --- | --- | --- | --- |
| A | Flux au pic | [0, ∞] | Flux au pic | [0, ∞] |
| t₀ | Temps au pic | [-∞, ∞] | Médiane(t₀) | Fixé |
| t_fall | 15 | [0, ∞] | Médiane(t_fall) | [1, ∞] |
| t_rise | 5 | [0, ∞] | Médiane(t_rise) | [1, ∞] |
| B | 0 | [-∞, ∞] | 0 | [-∞, ∞] |

## 5. Calcul des couleurs


### Méthode :

Les couleurs sont calculées comme la différence des magnitudes au pic entre deux bandes adjacentes :
c_ij = m_p,i - m_p,j  # Ex : c_ug = m_p,u - m_p,g

Si une bande a été ajustée par une constante (m_p = 0), la couleur est calculée avec une pondération par les erreurs (pour éviter des valeurs aberrantes).


### Normalisation :

Les flux sont normalisés par des facteurs de bande (ex : f_u = 1.67, f_g = 1.18, etc.) pour aligner les courbes avant de calculer t_max (voir Tableau 1 de votre chat précédent).



## 6. Coupures qualité post-ajustement
Pour garantir des fits robustes, l’article applique des coupures sur les paramètres de Bazin (Section 3.3) :

t_rise > 1 et t_rise non proche de 1 (tolérance = 0.01).
-20 < B < 20.
χ²/d.o.f < 10.
t_fall < 150.
t_rise < t_fall.
A < 5000 (et A(u), A(Y) < 1000).
Erreurs sur les paramètres : A_err < 100, t₀_err < 50, etc.
Impact :

Ces coupures éliminent ~85% des SNe CC (Type II) et ~45% des SNe Ia (car les CC ont souvent des t_fall très grands).
L’échantillon final pour la classification contient 68% de Ia, 20% de II, 11% de Ibc.


## 7. Implications pour votre travail (Fink, Rubin-LSST)

### À reproduire


#### Ajustement par bande :

Pour chaque SN et chaque bande (u, g, r, i, z, Y), ajuster la fonction de Bazin indépendamment.
Utiliser scipy.optimize.curve_fit avec les valeurs initiales et limites du Tableau 1.
Gérer les bandes à faible SNR avec un ajustement constant.


#### Calcul des features :

Extraire les 12 paramètres de Bazin (t_rise, t_fall pour chaque bande).
Calculer les 5 couleurs (c_ug, c_gr, c_ri, c_iz, c_zY).
Appliquer les coupures qualité pour filtrer les mauvais fits.


#### Classification :

Entraîner un Random Forest avec ces 17 features (12 paramètres + 5 couleurs).
Optimiser le seuil de probabilité pour atteindre la pureté souhaitée (ex : 99% pour la cosmologie).

#### Adaptations possibles


#### Optimisation pour Fink :

Prétraitement : Normaliser les courbes de lumière (ex : par la bande r, toujours présente).
Gestion des bandes manquantes : Si une bande n’a aucune mesure, l’ignorer dans le calcul des couleurs.
Pondération par SNR : Pour les bandes avec peu de points, utiliser une pondération par 1/σ_flux² (comme suggéré dans votre chat précédent).


#### Combinaison avec d’autres méthodes :

Utiliser SALT2 pour les SNe Ia (si le fit Bazin échoue ou pour les bas-z).
Intégrer des features supplémentaires (ex : paramètres de SALT2 comme x₁, c) pour améliorer la classification.


#### Performance :

Tester l’impact des coupures sur votre échantillon (ex : χ²/d.o.f < 10 peut être trop strict pour vos données).
Valider la pureté et l’efficacité sur des simulations réalistes (ex : ELASTICC).

## Exemple de code (Python)


In [ ]:
# --- Constantes globales ---
BANDS = ['u', 'g', 'r', 'i', 'z', 'Y']  # Bandes LSST
ZERO_POINT = 25.0  # Point zéro pour LSST (système AB)

# --- Fonction de Bazin ---
def bazin_function(t, A, t0, t_fall, t_rise, B):
    """Fonction de Bazin pour ajuster une courbe de lumière."""
    x = (t - t0) / t_rise
    return A * np.exp(-(t - t0) / t_fall) / (1 + np.exp(-x)) + B

# Exception personnalisée pour les erreurs d'ajustement
class BazinFitError(Exception):
    """Exception levée si l'ajustement de Bazin échoue."""
    pass
    

def fit_bazin_band(t, flux, flux_err, zero_point=ZERO_POINT):
    """
    Ajuste la fonction de Bazin sur une courbe de lumière d'une seule bande.
    Retourne un dictionnaire avec les paramètres, χ², et statut de succès.
    """
    # Vérifications des entrées
    if len(t) < 3:
        return {
            'success': False,
            'error': 'Pas assez de points (min 3 requis).',
            'params': None,
            'chi2': np.nan,
            'ndof': np.nan,
            'pcov': None
        }
    if np.any(flux_err <= 0):
        return {
            'success': False,
            'error': 'Erreurs sur le flux non strictement positives.',
            'params': None,
            'chi2': np.nan,
            'ndof': np.nan,
            'pcov': None
        }

    # Vérifications des entrées
    if len(t) != len(flux) or len(t) != len(flux_err):
        raise BazinFitError("Les tableaux t, flux et flux_err doivent avoir la même longueur.")
    if len(t) < 3:
        raise BazinFitError("Au moins 3 points de données sont requis pour l'ajustement.")
    if np.any(flux_err <= 0):
        raise BazinFitError("Les erreurs sur le flux doivent être strictement positives.")

    # Valeurs initiales (1ère étape)
    p0 = [np.max(flux), t[np.argmax(flux)], 15.0, 5.0, 0.0]
    bounds = ([0, -np.inf, 0, 0, -20], [np.inf, np.inf, np.inf, np.inf, 20])

    try:
        # Ajustement avec curve_fit
        popt, pcov = curve_fit(
            bazin_function,
            t,
            flux,
            p0=p0,
            bounds=bounds,
            sigma=flux_err,
            absolute_sigma=True
        )
        A, t0, t_fall, t_rise, B = popt

        # Calcul du χ²
        model_flux = bazin_function(t, *popt)
        chi2 = np.sum(((flux - model_flux) / flux_err) ** 2)
        ndof = len(t) - 5  # 5 paramètres : A, t0, t_fall, t_rise, B

        # Calcul de t_max et f_max (Eq. 3 et 4 de Dai et al. 2018)
        if t_fall / t_rise > 1:
            t_max = t0 + t_rise * np.log(t_fall / t_rise - 1)
            x = t_rise / t_fall
            f_max = A * (x ** x) * ((1 - x) ** (1 - x)) + B
            m_p = -2.5 * np.log10(f_max) + zero_point
        else:
            t_max, f_max, m_p = np.nan, np.nan, np.nan

        # Vérification des paramètres (coupures qualité)
        if (t_rise <= 1 or t_fall <= t_rise or A <= 0 or
            np.isnan(t_max) or np.isinf(t_max)):
            return {
                'success': False,
                'error': 'Paramètres non physiques (t_rise <= 1, t_fall <= t_rise, etc.).',
                'params': None,
                'chi2': chi2,
                'ndof': ndof,
                'pcov': pcov
            }

        # Stockage des paramètres
        return {
            'success': True,
            'params': {
                'A': A,
                't0': t0,
                't_fall': t_fall,
                't_rise': t_rise,
                'B': B,
                't_max': t_max,
                'f_max': f_max,
                'm_p': m_p
            },
            'chi2': chi2,
            'ndof': ndof,
            'pcov': pcov,
            'error': None
        }

    except RuntimeError as e:
        return {
            'success': False,
            'error': f"Échec de l'ajustement : {str(e)}",
            'params': None,
            'chi2': np.nan,
            'ndof': np.nan,
            'pcov': None
        }

def fit_single_event(ltcv_df: pd.DataFrame) -> dict:
    """
    Ajuste un événement (supernova) avec la fonction de Bazin pour chaque bande.
    Retourne un dictionnaire avec :
        - 'success': bool
        - 'global_params': dict (t_max global, F_peak global)
        - 'band_params': dict (paramètres de Bazin par bande)
        - 'colors': dict (couleurs calculées à partir des m_p)
        - 'chi2_total': float (χ² total)
        - 'ndof_total': int
        - 'error': str (si échec)
    """
    # --- Préparation des données ---
    mask_det = ltcv_df['FLUXCALERR'] > 0
    df = ltcv_df[mask_det].copy()
    if len(df) < 5:
        return {'success': False, 'error': 'Pas assez de points (min 5 requis).'}

    # --- Ajustement par bande ---
    band_results = {}
    chi2_total = 0.0
    ndof_total = 0
    successful_bands = []

    for band in BANDS:
        band_mask = df['BAND'] == band
        if band_mask.sum() < 3:
            band_results[band] = {
                'success': False,
                'error': f'Pas assez de points pour la bande {band}.',
                'params': None
            }
            continue

        t = df.loc[band_mask, 'MJD'].values
        flux = df.loc[band_mask, 'FLUXCAL'].values
        flux_err = df.loc[band_mask, 'FLUXCALERR'].values

        # Ajustement de Bazin pour cette bande
        result = fit_bazin_band(t, flux, flux_err)
        band_results[band] = result

        if result['success']:
            chi2_total += result['chi2']
            ndof_total += result['ndof']
            successful_bands.append(band)

    # --- Vérification qu'au moins une bande a réussi ---
    if not successful_bands:
        return {'success': False, 'error': 'Aucune bande n\'a pu être ajustée.'}

    # --- Calcul des paramètres globaux ---
    # t_max global : médiane des t_max des bandes réussies
    t_max_values = [
        band_results[band]['params']['t_max']
        for band in successful_bands
        if not np.isnan(band_results[band]['params']['t_max'])
    ]
    if not t_max_values:
        return {'success': False, 'error': 'Aucun t_max valide trouvé.'}

    t_max_global = np.median(t_max_values)

    # F_peak global : médiane des f_max des bandes réussies
    f_max_values = [
        band_results[band]['params']['f_max']
        for band in successful_bands
        if not np.isnan(band_results[band]['params']['f_max'])
    ]
    F_peak_global = np.median(f_max_values) if f_max_values else np.nan

    # --- Calcul des couleurs ---
    colors = {}
    # On calcule les couleurs entre bandes adjacentes (ex: u-g, g-r, etc.)
    for i in range(len(BANDS) - 1):
        band1 = BANDS[i]
        band2 = BANDS[i + 1]
        color_key = f'c_{band1}{band2}'
        m_p1 = band_results[band1]['params']['m_p'] if band_results[band1]['success'] else np.nan
        m_p2 = band_results[band2]['params']['m_p'] if band_results[band2]['success'] else np.nan
        if not (np.isnan(m_p1) or np.isnan(m_p2)):
            colors[color_key] = m_p1 - m_p2

    # --- Résultat final ---
    return {
        'success': True,
        'global_params': {
            't_max': t_max_global,
            'F_peak': F_peak_global,
        },
        'band_params': {band: band_results[band] for band in BANDS},
        'colors': colors,
        'chi2_total': chi2_total,
        'ndof_total': ndof_total,
        'chi2_red': chi2_total / max(ndof_total, 1),
        'error': None
    }

## 5 · Load data

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head  = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading truth for {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")

In [ ]:
# ── Filter: redshift + minimum detections ─────────────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)

truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()

print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

# Individual light-curve fits

### Random-preselection

In [ ]:
def filter_valid_events(subset, all_ltcvs, min_bands=MIN_BANDS, min_points=MIN_POINTS, min_total_points=MIN_TOTAL_POINTS):
    """
    Filtre les événements de `subset` qui ont :
    - Au moins `min_bands` bandes avec ≥ `min_points` points.
    - Au moins `min_total_points` points au total.

    Paramètres :
    -----------
    subset : pd.DataFrame
        DataFrame avec les métadonnées des événements (ex : SNID, ZCMB, etc.).
    all_ltcvs : pd.DataFrame
        DataFrame avec toutes les courbes de lumière (colonnes : SNID, MJD, FLUXCAL, FLUXCALERR, BAND).
    min_bands : int
        Nombre minimal de bandes avec ≥ `min_points` points.
    min_points : int
        Nombre minimal de points par bande.
    min_total_points : int
        Nombre minimal total de points (toutes bandes confondues).

    Retourne :
    --------
    valid_subset : pd.DataFrame
        Sous-ensemble de `subset` avec uniquement les événements valides.
    """
    valid_snids = []

    for snid in subset['SNID'].unique():
        # Récupérer les données de la courbe de lumière pour cet événement
        ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

        # Filtrer les points avec FLUXCALERR > 0 (détections valides)
        #ltcv = ltcv[ltcv['FLUXCALERR'] > 0]
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

        # Vérifier le nombre total de points
        if len(ltcv) < min_total_points:
            continue

        # Compter le nombre de bandes avec ≥ min_points points
        band_counts = ltcv['BAND'].value_counts()
        valid_bands = band_counts[band_counts >= min_points]
        if len(valid_bands) >= min_bands:
            valid_snids.append(snid)

    # Retourner le sous-ensemble filtré
    return subset[subset['SNID'].isin(valid_snids)]

In [ ]:
# ── Select N_CURVES events ────────────────────────────────────────────────────
#n_avail = min(N_CURVES, len(subset))
#chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
#chosen_snids = subset['SNID'].values[chosen_idx]
#print(f"Selected {n_avail} SNIDs for individual fitting.")

In [ ]:
# --- Filtrer les événements valides ---
valid_subset = filter_valid_events(
    subset,
    all_ltcvs,
    min_bands=MIN_BANDS,
    min_points=MIN_POINTS,
    min_total_points=MIN_TOTAL_POINTS
)

# Vérifier qu'il y a assez d'événements valides
n_avail = min(N_CURVES, len(valid_subset))
if n_avail == 0:
    raise ValueError("Aucun événement ne satisfait les critères de sélection.")

# Sélection aléatoire parmi les événements valides
chosen_idx = rng.choice(len(valid_subset), size=n_avail, replace=False)
chosen_snids = valid_subset['SNID'].values[chosen_idx]

print(f"Sélectionnés {n_avail} SNIDs valides pour l'ajustement individuel.")

In [ ]:
chosen_snids 

### Do the fit

In [ ]:
# --- Run individual fits ───────────────────────────────────────────────────────
fit_results_p1 = {}
idx = 0

for snid in chosen_snids:
    idx += 1
    print(f"\n ==== {idx}) SNID {snid:8d} ==== ")

    # Récupérer les données brutes
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

    # --- Debug : Avant filtre ---
    print(f"\n=== SNID {snid} : Before : Points par bande (total) ===")
    for band in BANDS:
        n_points = len(ltcv[ltcv['BAND'] == band])
        print(f"Bande {band}: {n_points} points")

    # Appliquer les filtres
    if DETECTED_ONLY:
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

    # Filtrer avec SNR > 3 et FLUXCAL > 0
    mask_det = (
        (ltcv['FLUXCALERR'] > 0) &
        (ltcv['FLUXCAL'] > 0) &
        (ltcv['FLUXCAL'] / ltcv['FLUXCALERR'] > 3)
    )
    ltcv = ltcv[mask_det]

    # --- Debug : Après filtre ---
    print(f"\n=== SNID {snid} : After : Points par bande ===")
    for band in BANDS:
        n_points = len(ltcv[ltcv['BAND'] == band])
        print(f"Bande {band}: {n_points} points")

    # Vérifier si ltcv est vide
    if len(ltcv) == 0:
        print(f"✗ SNID {snid}: Aucun point valide après filtre.")
        fit_results_p1[snid] = {'success': False, 'error': 'Aucun point valide.'}
        continue  # Passer au SNID suivant

    # Appel de fit_single_event
    try:
        result = fit_single_event(ltcv)
    except Exception as e:
        print(f"✗ SNID {snid}: Erreur dans fit_single_event - {str(e)}")
        fit_results_p1[snid] = {'success': False, 'error': f'Erreur: {str(e)}'}
        continue  # Passer au SNID suivant

    # Affichage des résultats
    if result['success']:
        print("\n=== Résultats globaux ===")
        print(f"t_max global: {result['global_params']['t_max']:.2f}")
        print(f"F_peak global: {result['global_params']['F_peak']:.2f}")
        print(f"χ²/ndof total: {result['chi2_red']:.2f}")
        print(f"Couleurs: {result['colors'] if result['colors'] else 'Aucune'}")

        print("\n=== Paramètres par bande ===")
        for band in BANDS:
            band_result = result['band_params'][band]
            if band_result['success']:
                params = band_result['params']
                print(f"\n  ✓ Bande {band}:")
                print(f"    A: {params['A']:.2f}, t0: {params['t0']:.2f}, t_fall: {params['t_fall']:.2f}, t_rise: {params['t_rise']:.2f}")
                print(f"    t_max: {params['t_max']:.2f}, f_max: {params['f_max']:.2f}, m_p: {params['m_p']:.2f}")
                print(f"    χ²/ndof: {band_result['chi2'] / band_result['ndof']:.2f}")
            else:
                print(f"\n  ✗ Bande {band}: {band_result['error']}")
    else:
        print(f"\n✗ SNID {snid}: Échec global - {result['error']}")

    # Stocker le résultat (même en cas d'échec)
    fit_results_p1[snid] = result

    # Affichage résumé (avec gestion des KeyError)
    status = '✓' if result.get('success') else '✗'
    chi2r = result.get('chi2_red', float('nan'))
    t_max = result.get('global_params', {}).get('t_max', np.nan)  # Gestion de KeyError
    print(f"\n  SNID {snid:8d}  {status}  χ²/dof = {chi2r:.2f}  t_max={t_max:.1f}")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# --- Constantes pour le traçage ---
FIG_W, FIG_H = 5.5, 4.2
NCOLS = 3  # Nombre de colonnes dans la grille de subplots
BAND_COLORS = {
    'u': 'purple', 'g': 'blue', 'r': 'green',
    'i': 'orange', 'z': 'red', 'Y': 'brown'
}

def plot_bazin_fits(chosen_snids, fit_results_p1, all_ltcvs, truth,
                   DETECTED_ONLY=False, esr=None, Z_MIN=0, Z_MAX=1.2,
                   OBJ_CLASS="SNIa", MIN_DETECTIONS=5):
    """
    Trace les courbes de lumière et les fits Bazin pour chaque SNID.

    Paramètres :
    -----------
    chosen_snids : list
        Liste des SNID à tracer.
    fit_results_p1 : dict
        Dictionnaire des résultats de fit_single_event (clé = SNID).
    all_ltcvs : pd.DataFrame
        DataFrame contenant toutes les courbes de lumière.
    truth : pd.DataFrame
        DataFrame avec les vraies valeurs (ex : ZCMB).
    DETECTED_ONLY : bool
        Si True, ne trace que les points avec PHOTFLAG détecté.
    esr : module
        Module contenant esr.photflag_detect (pour DETECTED_ONLY).
    """
    n_avail = len(chosen_snids)
    nrows = math.ceil(n_avail / NCOLS)

    fig, axes = plt.subplots(
        nrows, NCOLS,
        figsize=(FIG_W * NCOLS, FIG_H * nrows),
        tight_layout=True
    )
    axes_flat = np.array(axes).flatten()

    # Temps dense pour le modèle Bazin (en jours relatifs à t_max)
    t_model_dense = np.linspace(-60, 100, 500)

    for idx, snid in enumerate(chosen_snids):
        ax = axes_flat[idx]
        res = fit_results_p1[snid]

        # Récupérer les données de la SN
        ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]
        if DETECTED_ONLY and esr is not None:
            ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

        # Récupérer le redshift (si disponible)
        z_val = truth[truth['SNID'] == snid]['ZCMB'].values
        z_val = z_val[0] if len(z_val) else float('nan')

        # Vérifier si le fit a réussi
        if not res.get('success', False):
            ax.set_title(f"SNID {snid}\nFit échoué", fontsize=9, color='red')
            ax.set_visible(False)  # Masquer le subplot si échec
            continue

        # Récupérer les paramètres globaux
        t_max_global = res['global_params']['t_max']
        F_peak_global = res['global_params']['F_peak']
        chi2r = res.get('chi2_red', float('nan'))

        # --- Tracer les données observées (renormalisées) ---
        for band in BANDS:
            bdf = ltcv[ltcv['BAND'] == band]
            if len(bdf) == 0:
                continue

            # Récupérer les paramètres Bazin pour cette bande
            band_result = res['band_params'][band]
            if not band_result['success']:
                continue  # Sauter si l'ajustement a échoué

            params = band_result['params']
            A_b = params['A']
            norm_factor = max(A_b * F_peak_global, 1e-6)  # Normalisation pour échelle commune

            # Temps relatif à t_max_global
            dt = bdf['MJD'].values - t_max_global
            f_n = bdf['FLUXCAL'].values / norm_factor
            fe_n = bdf['FLUXCALERR'].values / norm_factor

            # Tracer les points observés
            ax.errorbar(
                dt, f_n, yerr=fe_n,
                color=BAND_COLORS.get(band, 'gray'),
                linestyle='None', marker='o', markersize=3,
                capsize=2, label=band
            )

            # --- Tracer le fit Bazin pour cette bande ---
            # Générer la courbe de Bazin sur t_model_dense + t_max_global
            t_abs = t_model_dense + t_max_global
            flux_model = bazin_function(
                t_abs,
                params['A'], params['t0'], params['t_fall'], params['t_rise'], params['B']
            )
            # Normaliser la courbe modèle
            flux_model_n = flux_model / norm_factor
            ax.plot(
                t_model_dense, flux_model_n,
                color=BAND_COLORS.get(band, 'gray'),
                linestyle='-', linewidth=1.5,
                label=f'Bazin {band}'
            )

        # --- Configuration du subplot ---
        ax.axhline(0, color='k', lw=0.4, ls='--')
        ax.set_xlim(t_model_dense[0], t_model_dense[-1])
        ax.set_ylim(0, 1.2)  # Ajustez selon vos données

        # Titre avec infos
        title = (
            f"SNID {snid}  z={z_val:.3f}\n"
            f"t_max={t_max_global:.1f}  F_peak={F_peak_global:.1f}  χ²/dof={chi2r:.2f}"
        )
        ax.set_title(title, fontsize=8)

        ax.set_xlabel(r"$t - t_{\rm max}$ [days]", fontsize=8)
        ax.set_ylabel(r"Flux / $(A_b F_{\rm peak})$", fontsize=8)
        ax.tick_params(axis='both', labelsize=7)
        ax.legend(fontsize=6, ncol=2, loc='upper right')

    # Masquer les subplots vides
    for idx in range(n_avail, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Titre global
    fig.suptitle(
        f"Fits Bazin par bande — {OBJ_CLASS}\n"
        f"z ∈ [{Z_MIN},{Z_MAX})",
        fontsize=11, y=1.01
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# --- Constantes pour le traçage ---
FIG_W, FIG_H = 5.5, 4.2
NCOLS = 3  # Nombre de colonnes dans la grille
BAND_COLORS = {
    'u': 'purple', 'g': 'blue', 'r': 'green',
    'i': 'orange', 'z': 'red', 'Y': 'brown'
}

def plot_bazin_fits(chosen_snids, fit_results_p1, all_ltcvs, truth,
                   DETECTED_ONLY=False, esr=None, Z_MIN=0, Z_MAX=1.2,
                   OBJ_CLASS="SNIa", MIN_DETECTIONS=5):
    """
    Trace les courbes de lumière et les fits Bazin pour chaque SNID.
    """
    n_avail = len(chosen_snids)
    nrows = math.ceil(n_avail / NCOLS)

    # Créer la figure avec des marges ajustées
    fig, axes = plt.subplots(
        nrows, NCOLS,
        figsize=(FIG_W * NCOLS, FIG_H * nrows),
        tight_layout=True
    )
    axes_flat = np.array(axes).flatten()

    # Temps dense pour le modèle Bazin (en jours relatifs à t_max)
    t_model_dense = np.linspace(-60, 100, 500)

    for idx, snid in enumerate(chosen_snids):
        ax = axes_flat[idx]
        res = fit_results_p1[snid]

        # Récupérer les données de la SN
        ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]
        if DETECTED_ONLY and esr is not None:
            ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

        # Récupérer le redshift (si disponible)
        z_val = truth[truth['SNID'] == snid]['ZCMB'].values
        z_val = z_val[0] if len(z_val) else float('nan')

        # Vérifier si le fit a réussi
        if not res.get('success', False):
            ax.set_title(f"SNID {snid}\nFit échoué", fontsize=8, color='red')
            ax.set_visible(False)
            continue

        # Récupérer les paramètres globaux
        t_max_global = res['global_params']['t_max']
        F_peak_global = res['global_params']['F_peak']
        chi2r = res.get('chi2_red', float('nan'))

        # --- Tracer les données observées et les fits Bazin ---
        ymin, ymax = float('inf'), -float('inf')  # Pour ajuster l'axe Y

        for band in BANDS:
            bdf = ltcv[ltcv['BAND'] == band]
            if len(bdf) == 0:
                continue

            band_result = res['band_params'][band]
            if not band_result['success']:
                continue  # Sauter si l'ajustement a échoué

            params = band_result['params']
            A_b = params['A']
            norm_factor = max(A_b * F_peak_global, 1e-6)

            # Temps relatif à t_max_global
            dt = bdf['MJD'].values - t_max_global
            f_n = bdf['FLUXCAL'].values / norm_factor
            fe_n = bdf['FLUXCALERR'].values / norm_factor

            # Mettre à jour ymin/ymax pour l'axe Y
            ymin = min(ymin, np.min(f_n - fe_n))
            ymax = max(ymax, np.max(f_n + fe_n))

            # Tracer les points observés
            ax.errorbar(
                dt, f_n, yerr=fe_n,
                color=BAND_COLORS.get(band, 'gray'),
                linestyle='None', marker='o', markersize=3,
                capsize=2, label=band
            )

            # Tracer le fit Bazin
            t_abs = t_model_dense + t_max_global
            flux_model = bazin_function(
                t_abs, params['A'], params['t0'], params['t_fall'], params['t_rise'], params['B']
            )
            flux_model_n = flux_model / norm_factor
            ax.plot(
                t_model_dense, flux_model_n,
                color=BAND_COLORS.get(band, 'gray'),
                linestyle='-', linewidth=1.5,
                label=f'Bazin {band}'
            )

        # --- Ajuster les limites de l'axe Y ---
        if ymin == float('inf') or ymax == -float('inf'):
            # Cas où aucune bande n'a été tracée (ne devrait pas arriver)
            ymin, ymax = 0, 1.2
        else:
            # Ajouter une marge de 10% en haut et en bas
            margin = 0.1 * (ymax - ymin)
            ymin -= margin
            ymax += margin
            # Limiter ymin à 0 (les flux sont positifs)
            ymin = max(ymin, 0)

        ax.set_ylim(ymin, ymax)

        # --- Configuration du subplot ---
        ax.axhline(0, color='k', lw=0.4, ls='--')
        ax.set_xlim(t_model_dense[0], t_model_dense[-1])

        # Titre avec infos (centré et ajusté)
        title = (
            f"SNID {snid}\n"
            f"z={z_val:.3f}  t_max={t_max_global:.1f}  "
            f"F_peak={F_peak_global:.1f}  χ²/dof={chi2r:.2f}"
        )
        ax.set_title(title, fontsize=8, pad=6)  # `pad` pour espacer le titre

        ax.set_xlabel(r"$t - t_{\rm max}$ [days]", fontsize=8)
        ax.set_ylabel(r"Flux / $(A_b F_{\rm peak})$", fontsize=8)
        ax.tick_params(axis='both', labelsize=7)

        # Légende en haut à droite (ajustée pour éviter le dépassement)
        if len(ax.get_legend_handles_labels()[0]) > 0:  # Si des éléments à légender
            ax.legend(
                fontsize=6,
                ncol=2,
                loc='upper right',
                bbox_to_anchor=(1.0, 1.0)  # Ajuste la position de la légende
            )

    # Masquer les subplots vides
    for idx in range(n_avail, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Titre global (ajusté pour ne pas dépasser)
    fig.suptitle(
        f"Fits Bazin par bande — {OBJ_CLASS}\n"
        f"z ∈ [{Z_MIN},{Z_MAX})",
        fontsize=10,  # Réduit pour éviter le dépassement
        y=1.02,       # Ajuste la position verticale
        x=0.5        # Centre horizontalement
    )

    # Ajustement final des marges
    plt.tight_layout(rect=[0, 0, 1, 0.98])  # `rect` pour laisser de la place au titre global
    plt.show()

In [ ]:
# Supposons que vous avez :
# - chosen_snids : liste des SNID à tracer (ex : [57899949, 57899950, ...])
# - fit_results_p1 : dictionnaire des résultats de fit_single_event
# - all_ltcvs : DataFrame avec toutes les courbes de lumière
# - truth : DataFrame avec les vraies valeurs (ex : ZCMB)

# Appel de la fonction de traçage
plot_bazin_fits(
    chosen_snids=chosen_snids,
    fit_results_p1=fit_results_p1,
    all_ltcvs=all_ltcvs,
    truth=truth,
    DETECTED_ONLY=True,
    esr=esr,  # Module contenant esr.photflag_detect
    OBJ_CLASS="SNIa",
    MIN_DETECTIONS=5,
)